# G2b · TANL's word corpus and the Four-OOD streams (checklist 2.3)

Seam 2 now runs on the exported streams. To build ReNeg **on top of TANL** (and to compare on TANL's second
benchmark, Four-OOD) the local analysis needs two more things, exported to `MyDrive/ReNeg/transfers/analysis_pack2/`:

1. `reneg_textbank_*.npz` — the CLIP text embeddings of TANL's whole 69,554-word corpus (official order, 8-bit),
   with the words, NegLabel's 10,000-word split and the 15 noise-image features TANL starts from.
2. `reneg_stream4ood_*.npz` — ImageNet val (50,000, the ID set of Four-OOD), SUN, Places and Textures (all), 8-bit.

**Runtime:** CPU is fine, about 3 minutes. Then attach every file the last cell lists (each under 25 MB).

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish
ctx = start("G2b_textbank_and_four_ood", need_gpu=False)
log = ctx.log
import numpy as np, torch
from oodlab.report import find_cache
from reneg.packs import export_streams, export_textbank, load_textbank_export
BACKBONE = "ViT-B/16"
OUT = f"{DRIVE_ROOT}/transfers/analysis_pack2"
os.makedirs(OUT, exist_ok=True)
cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
print(bank.summary())

## 1 · TANL's corpus (8-bit) and a round-trip check

In [ ]:
tb_paths = export_textbank(bank, OUT, max_mb=24.0, log=log.info)
d = load_textbank_export(tb_paths)
ref = torch.nn.functional.normalize(bank.corpus_text.float(), dim=1)
rows = torch.randperm(ref.shape[0], generator=torch.Generator().manual_seed(0))[:2000]
err = float((d["corpus_text"][rows] @ ref[rows].T - ref[rows] @ ref[rows].T).abs().max())
print(f"{d['corpus_text'].shape[0]} corpus rows; largest cosine error after 8-bit coding: {err:.5f} (fine below 0.005)")

## 2 · Four-OOD streams (8-bit)

In [ ]:
SETS = ["imagenet_val_all", "sun", "places", "textures_all"]
print({s: cache.has(s) for s in SETS})
four_paths = export_streams(cache, OUT, SETS, max_mb=24.0, prefix="reneg_stream4ood", log=log.info)

## Files written

In [ ]:
files = tb_paths + four_paths
total = 0
for p in files:
    mb = os.path.getsize(p) / 1e6
    total += mb
    print(f"{mb:6.1f} MB  {p}")
print(f"{len(files)} files, {total:.0f} MB in total, all in {OUT}")
finish(ctx, {"files": [os.path.basename(p) for p in files]})

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")